<a href="https://colab.research.google.com/github/leonardomenezes10/Fundamentos/blob/main/notebooks/MRE/MRE_Notas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introdução ao Pensamento computacional

## Ferramentas/Serviços utilizados

- Google Colab (notebook com codigo)
- Github (rede social de código)
- Git (programa de versionamento de código)


## Pilares
- Decomposição
- Abstração
- Identificação de padrões
- Lógica


## Noções Gerais

- Exemplos a partir de coleta de dados
  - Site MRE - Notas de imprensa
    - [x] Link do site: https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/notas-a-imprensa
    - [x] Entender a estrutura da fonte de informação
    - [ ] Realizar a coleta
    - [ ] Inserir as informações em um banco de dados
    - [ ] Utilizar as informações (analise de dados)


### Padrão de paginação das notas de imprensa
  - https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/notas-a-imprensa?b_start:int=0
   - https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/notas-a-imprensa?b_start:int=30
   - Pagina final (10/02/2026): 6210



## Importação de bibliotecas/pogramas utilizados neste arquivo

In [ ]:
!pip install tinydb

In [ ]:
# programas/bibliotecas utilizados no script/codigo
import httpx # Responsável pelas requisições web
from bs4 import BeautifulSoup # Responsável por realizar o web scraping (coletar os dados)
from tinydb import TinyDB, Query

## Criação do banco json

In [ ]:
def inserir_no_banco(dados, link_noticia):
  arquivo_banco_dados = "nota_mre.json"
  db = TinyDB(arquivo_banco_dados)


  # Evitar dados repetidos no banco
  Buscar = Query()
  verificar_link = db.contains(Buscar.link == link_noticia)

  if not verificar_link:
    print("Inserindo nova informação no banco")
    db.insert(dados)
  else:
    print("Link já existe no banco. Esta informação não será inserida novamente")

In [ ]:
# Variável e tipos de dados (string, lista, numero)
all_paginas_links = [] # Initialize as an empty list
for x in range (0, 1240, 30):
  url = (f'https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/notas-a-imprensa?b_start:int={x}')
  all_paginas_links.append(url) # Add each URL to the list

def acessa_pagina (link):
  print (f"Estamos na pagina:{link}")

  # Define headers para a requisição, simulando um navegador
  headers = {
      'User-Agent': "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36",
      'Accept-Language': 'en-US,en;q=0.9',
      'Accept-Encoding': 'gzip, deflate, br',
      'Connection': 'keep-alive',
  }

  timeout = httpx.Timeout(connect=20.0, read=30.0, write=20.0, pool=10.0)
  pag_web = httpx.get(link, headers=headers, timeout=timeout)
  bs = BeautifulSoup(pag_web, "html.parser")
  return bs

# loop for
# beautifulsoap >> find e find_all

for pagina in all_paginas_links:
  pagina_inteira = acessa_pagina(pagina)
  lista_noticias = pagina_inteira.find("div", attrs={"id": "content-core"}).find_all("article")
  for noticia in lista_noticias:
    # titulo
    try:
      titulo = noticia.find("h2", attrs={"class": "tileHeadline"}).text.strip()
      print(titulo)
    except:
        titulo = ""

    #link
    try:
      link_noticia = noticia.a["href"]
      print(link_noticia)
    except:
      link_noticia = ""
    # numero da nota - exemplo: NOTA À IMPRENSA Nº 72
    # numero da nota - exemplo: NOTA À IMPRENSA N° 590
    num_nota = noticia.find("span", attrs={"class": "subtitle"}).text.strip()
    # print(num_nota)
    # num_nota = noticia.find(attrs={"class": "subtitle"}).text.strip()
    num_nota = num_nota.replace("NOTA À IMPRENSA N°", "").replace("NOTA À IMPRENSA Nº", "").strip()
    print(num_nota)
    print("###")
    # data
    # horário
    data_hora = noticia.find_all("span",attrs={"class": "summary-view-icon"})
    data= data_hora[0].text.strip()
    hora = data_hora[1].text.strip()
    print(data)
    print(hora)
    conteudo = acessa_pagina (link_noticia)
    paragrafos = conteudo.find("div", attrs={"property":"rnews:articleBody"}).find_all("p")
    lista_paragrafos = []
    for paragrafo in paragrafos:
      lista_paragrafos.append(paragrafo.text.strip())
    print(lista_paragrafos)
    # função para inserir dados coletados no banco
    dados = {
        "titulo": titulo,
        "link": link_noticia,
        "data": data,
        "hora": hora,
        "num_nota": num_nota,
        "paragrafo": lista_paragrafos
    }
    inserir_no_banco(dados,link_noticia)






# Transformar banco json e dataframe

- pre-analise - entendimento geral sobre o dataframe

In [ ]:
import pandas as pd
import json

## Abrindo o rquivo json
with open("nota_mre.json") as f:
  raw = json.load(f)

df = pd.DataFrame.from_dict(raw["_default"], orient="index")

df


In [ ]:
# saber quantidade de linhas e colunas do dataframe
df.shape

In [ ]:
# saber colunas disponiveis
df.columns

In [ ]:
# selecionar uma coluna em especifico
df["titulo"]

In [ ]:

# delimitar colunas do dataframe
df_delimitado = df[["titulo", "data"]]
df_delimitado

In [ ]:

# primeiras (head), ultimas (tail) e linhas aleatórias (sample)
df.head(10)

In [ ]:
df.describe(include="all")

In [ ]:
df.isnull().sum()

### Tratamento de Dados e Visualização

Agora que entendemos a estrutura, vamos preparar os dados para extrair insights visuais. Um passo comum é converter colunas de texto em formatos que o computador entenda como 'tempo' (datetime) e criar gráficos.

In [ ]:
# 1. Converter a coluna de data para o formato datetime do Pandas
df['data_dt'] = pd.to_datetime(df['data'], dayfirst=True)

# 2. Contar quantas notícias temos por dia
noticias_por_dia = df['data_dt'].value_counts().sort_index()

display(noticias_por_dia)

#### Visualizando o Volume de Publicações

Vamos usar a biblioteca `matplotlib` (que já vem no ambiente) para criar um gráfico de barras simples.

In [ ]:
import matplotlib.pyplot as plt

# Criando o gráfico
plt.figure(figsize=(15, 10))
noticias_por_dia.plot(kind='bar', color='red')

# Adicionando títulos e rótulos
plt.title('Quantidade de Notas à Imprensa por Data')
plt.xlabel('Data da Publicação')
plt.ylabel('Número de Notas')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

#### Analisando Palavras-Chave nos Títulos

Uma técnica simples de análise de texto para iniciantes é verificar a frequência de certas palavras-chave (como nomes de países).

In [ ]:
paises = ['China', 'Estados Unidos', 'Europa', 'Inteligência Artificial', 'Latina', ' IA ']
frequencia = {}

for pais in paises:
    # Conta em quantos títulos a palavra aparece
    frequencia[pais] = df['titulo'].str.contains(pais, case=False).sum()

# Converter para série para facilitar a plotagem
ser_freq = pd.Series(frequencia)

plt.figure(figsize=(8, 4))
ser_freq.sort_values(ascending=False).plot(kind='barh', color='lightgreen')
plt.title('Menções de Países nos Títulos das Notas')
plt.xlabel('Número de Menções')
plt.show()

---

# 📊 Visualização avançada dos dados

Nesta parte vamos:

1. **Preparar** os dados (corrigir tipos, criar colunas auxiliares)
2. **Buscar** por termos e palavras-chave (no título e no corpo da nota)
3. **Analisar** frequência de termos (países, temas, organizações)
4. **Visualizar** o volume de publicações no tempo
5. **Acompanhar** um tema específico ao longo do tempo
6. **Gerar** nuvens de palavras (word cloud)
7. **Examinar** o tamanho das notas
8. **Descobrir** co-ocorrências (quais países aparecem juntos)


## 1. Preparação dos dados

Antes de visualizar, precisamos arrumar algumas coisas:

- A coluna `data` está como **texto** (`"22/04/2026"`). Vamos transformar em **datetime** para conseguir agrupar por mês, dia da semana, etc.
- A coluna `paragrafo` contém **listas** de strings — por isso `df.duplicated()` deu erro mais acima. Vamos juntar os parágrafos em um único texto na coluna `texto`.
- Vamos criar uma coluna `texto_completo` com título + corpo, em minúsculas, para facilitar buscas case-insensitive.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1.1 — Converter data para datetime (dayfirst pq é formato brasileiro: dd/mm/aaaa)
df['data_dt'] = pd.to_datetime(df['data'], dayfirst=True, errors='coerce')

# 1.2 — Juntar os parágrafos (que são listas) em um único texto
df['texto'] = df['paragrafo'].apply(lambda lista: ' '.join(lista) if isinstance(lista, list) else str(lista))

# 1.3 — Coluna com tudo junto e em minúsculas (facilita buscas)
df['texto_completo'] = (df['titulo'].fillna('') + ' ' + df['texto']).str.lower()

# 1.4 — Extrair a hora como número inteiro (de "19h20" tira o 19)
df['hora_int'] = df['hora'].str.extract(r'(\d{1,2})').astype(float)

# 1.5 — Colunas auxiliares de tempo
df['ano_mes'] = df['data_dt'].dt.to_period('M')
df['dia_semana'] = df['data_dt'].dt.day_name()
df['tamanho_texto'] = df['texto'].str.len()
df['qtd_paragrafos'] = df['paragrafo'].apply(lambda x: len(x) if isinstance(x, list) else 0)

print(f"Total de notas: {len(df)}")
print(f"Período: de {df['data_dt'].min().date()} até {df['data_dt'].max().date()}")
df[['titulo', 'data_dt', 'hora_int', 'qtd_paragrafos', 'tamanho_texto']]

# customização de intervalo de datas
data_custom = "2026-05-22"
print(f"Período: de {data_custom} até {df['data_dt'].max().date()}")
df_custom = df[df['data_dt'] >= data_custom]
df_custom


### 1.1 Verificar duplicatas

Como a coluna `paragrafo` é uma lista, ela quebra o `df.duplicated()`. Solução: verificar duplicatas só nas colunas que importam (link, título).


In [ ]:
df

In [ ]:
# Duplicatas por link (cada nota deveria ter um link único)
print("Duplicatas por link:", df.duplicated(subset=['link']).sum())

# Duplicatas por título
print("Duplicatas por título:", df.duplicated(subset=['titulo']).sum())


## 2. 🔍 Busca por termos e palavras-chave

A busca anterior só olhava o **título**. Vamos criar uma função que busca no **título e/ou no corpo** da nota, mostrando quantas notas mencionam o termo e permitindo ver os resultados.


In [ ]:
def buscar_termo(termo, onde='completo', mostrar=5):
    """
    Busca um termo nas notas do MRE.

    Parâmetros:
        termo (str): palavra ou expressão a buscar (não diferencia maiúsc/minúsc)
        onde (str): 'titulo', 'texto' ou 'completo' (título + corpo)
        mostrar (int): quantos resultados imprimir

    Retorna:
        DataFrame com as notas que contêm o termo
    """
    coluna = {
        'titulo': 'titulo',
        'texto': 'texto',
        'completo': 'texto_completo',
    }[onde]

    # busca case-insensitive; na=False ignora valores faltantes
    mascara = df[coluna].str.contains(termo, case=False, na=False, regex=False)
    resultado = df[mascara].sort_values('data_dt', ascending=False)

    print(f"🔎 Termo: '{termo}'  |  Onde: {onde}")
    print(f"📌 Notas encontradas: {len(resultado)} de {len(df)} ({len(resultado)/len(df)*100:.1f}%)")
    print("-" * 60)

    for _, row in resultado.head(mostrar).iterrows():
        data_str = row['data_dt'].strftime('%d/%m/%Y') if pd.notna(row['data_dt']) else '?'
        print(f"  • [{data_str}] {row['titulo']}")

    if len(resultado) > mostrar:
        print(f"  ... e mais {len(resultado) - mostrar} nota(s)")

    return resultado


# Exemplos de uso:
res = buscar_termo('China')
res = buscar_termo(' IA ')
res = buscar_termo('Inteligência Artificial')

## 3. 📈 Frequência de vários termos comparados

Vamos generalizar a análise: em vez de fixar 5 países, comparamos qualquer lista de termos de interesse (países, organizações, temas) — buscando no texto **completo** (não só no título).


In [ ]:
def frequencia_termos(lista_termos, onde='completo'):
    """Conta em quantas notas cada termo da lista aparece."""
    coluna = {'titulo': 'titulo', 'texto': 'texto', 'completo': 'texto_completo'}[onde]
    freq = {}
    for termo in lista_termos:
        freq[termo] = df[coluna].str.contains(termo, case=False, na=False, regex=False).sum()
    return pd.Series(freq).sort_values(ascending=True)


# Lista mais ampla de países e temas
termos_interesse = [
    'Inteligência Artificial', ' IA ', 'Digital', 'Tecnologia', 'Governança',
    'China', 'Europa', 'Estados Unidos', 'Latina', 'Japão',
    'Alemanha', 'CELAC', 'BRICS', 'Gateway', 'CEPAL'
]

freq = frequencia_termos(termos_interesse, onde='completo')
freq
# Gráfico
fig, ax = plt.subplots(figsize=(8, 6))
freq.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Menções por país/região no texto completo das notas', fontsize=13)
ax.set_xlabel('Número de notas')
ax.grid(axis='x', linestyle='--', alpha=0.5)

# Anotar valores nas barras
for i, valor in enumerate(freq.values):
    ax.text(valor + 0.1, i, str(valor), va='center')

plt.tight_layout()
plt.show()


## 4. 🧠 Extração automática de palavras mais frequentes

Em vez de **definirmos** os termos, deixamos os dados falarem: quais palavras aparecem mais nos títulos? Para isso precisamos remover as **stopwords** (palavras muito comuns como "de", "da", "para") e palavras curtas.


In [ ]:
import re
from collections import Counter

# Stopwords em português (lista pequena, suficiente para começar)
STOPWORDS_PT = {
    'a', 'o', 'as', 'os', 'um', 'uma', 'uns', 'umas',
    'de', 'do', 'da', 'dos', 'das', 'em', 'no', 'na', 'nos', 'nas',
    'e', 'ou', 'mas', 'que', 'se', 'por', 'para', 'com', 'sem',
    'à', 'ao', 'às', 'aos', 'pelo', 'pela', 'pelos', 'pelas',
    'é', 'são', 'foi', 'ser', 'estar', 'tem', 'ter', 'há',
    'sobre', 'entre', 'até', 'após', 'pela', 'pelo',
    'sua', 'seu', 'suas', 'seus', 'este', 'esta', 'isso', 'esse', 'essa',
    'nº', 'n°', 'nota', 'notas', 'imprensa',  # específicas do contexto
}


def palavras_mais_frequentes(serie_texto, top_n=20, min_tamanho=4):
    """Conta as palavras mais frequentes em uma coluna de texto."""
    todas_palavras = []
    for texto in serie_texto.dropna():
        # \w+ pega sequências de letras/números; flags re.UNICODE pra acentos
        palavras = re.findall(r'\b[a-záàâãéêíóôõúüç]+\b', texto.lower(), flags=re.UNICODE)
        palavras = [p for p in palavras if p not in STOPWORDS_PT and len(p) >= min_tamanho]
        todas_palavras.extend(palavras)
    return Counter(todas_palavras).most_common(top_n)


top_palavras = palavras_mais_frequentes(df['titulo'], top_n=20)
print("Top 20 palavras nos TÍTULOS:")
for palavra, contagem in top_palavras:
    print(f"  {palavra:25s} {contagem}")

# Gráfico
serie = pd.Series(dict(top_palavras)).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(10, 7))
serie.plot(kind='barh', ax=ax, color='coral')
ax.set_title('Top 20 palavras mais frequentes nos títulos das notas', fontsize=13)
ax.set_xlabel('Frequência')
ax.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


## 5. 🗓️ Análise temporal

O gráfico anterior ("notas por dia") fica difícil de ler quando há muitas datas. Vamos olhar em **agregações** mais úteis: por mês, por dia da semana, e por hora do dia.


In [ ]:
# Notas por mês
notas_por_mes = df.groupby('ano_mes').size()

notas_por_mes

fig, ax = plt.subplots(figsize=(11, 4))
notas_por_mes.plot(kind='bar', ax=ax, color='teal')
ax.set_title('Volume de notas por mês', fontsize=13)
ax.set_xlabel('Ano-Mês')
ax.set_ylabel('Número de notas')
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# Notas por dia da semana (em português)
ordem_dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
traducao_dias = {
    'Monday': 'Segunda', 'Tuesday': 'Terça', 'Wednesday': 'Quarta',
    'Thursday': 'Quinta', 'Friday': 'Sexta', 'Saturday': 'Sábado', 'Sunday': 'Domingo'
}

por_dia_semana = df['dia_semana'].value_counts().reindex(ordem_dias).fillna(0)
por_dia_semana.index = [traducao_dias[d] for d in por_dia_semana.index]

fig, ax = plt.subplots(figsize=(9, 4))
por_dia_semana.plot(kind='bar', ax=ax, color='mediumpurple')
ax.set_title('Em qual dia da semana o MRE mais publica notas?', fontsize=13)
ax.set_ylabel('Número de notas')
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Notas por hora do dia
por_hora = df['hora_int'].dropna().astype(int).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(11, 4))
por_hora.plot(kind='bar', ax=ax, color='goldenrod')
ax.set_title('Em qual hora do dia as notas são publicadas?', fontsize=13)
ax.set_xlabel('Hora (0–23)')
ax.set_ylabel('Número de notas')
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 6. 📅 Acompanhar um tema ao longo do tempo

Combinando busca por termo + análise temporal: como evoluiu a menção a um país ou tema ao longo dos meses?


In [ ]:
def evolucao_termo(termo, onde='completo'):
    """Mostra quantas notas mencionam o termo por mês."""
    coluna = {'titulo': 'titulo', 'texto': 'texto', 'completo': 'texto_completo'}[onde]
    mascara = df[coluna].str.contains(termo, case=False, na=False, regex=False)
    return df[mascara].groupby('ano_mes').size()


# Comparar evolução de vários termos no tempo
termos_comparar = ['inteligência artificial', 'China', 'Europa', 'digital', 'tecnologia', 'governança']

fig, ax = plt.subplots(figsize=(11, 5))
for termo in termos_comparar:
    serie = evolucao_termo(termo)
    if len(serie) > 0:
        serie.plot(ax=ax, marker='o', label=termo)

# Eventos diplomáticos-chave
for data_evento, rotulo in [
    ('2023-03-12', '12-15 Abr/2023\n3ª Visita de Estado de Lula à China'),
    ('2023-08-24', '24 Ago/2023\nReunião COSBAN preparatória'),
    ('2024-06-04', '4-7 Jun/2024\nVII Sessão Plenária da COSBAN'),
    ('2024-11-05', '19-20 Nov/2024\nVisita de Estado de Xi Jinping ao Brasil'),
    ('2025-04-01', '10-13 Mai/2025\n4ª Visita de Estado de Lula à China\n+ Fórum BRICS sobre IA 2025'),
]:
    evento = pd.Timestamp(data_evento)
    ax.axvline(evento, color='gray', linestyle='--', linewidth=1.3, alpha=0.8)
    ax.text(evento, ax.get_ylim()[1], rotulo, rotation=90, va='top', ha='right', fontsize=10, color='gray')

ax.set_title('Menções por mês (no texto completo das notas)', fontsize=13)
ax.set_xlabel('Ano-Mês')
ax.set_ylabel('Número de notas que mencionam o termo')
ax.legend()
ax.grid(linestyle='--', alpha=0.5)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
def obter_serie_bloco(lista_termos, freq='Q'):
    """Retorna a contagem de termos agrupada por tempo."""
    padrao = '|'.join(lista_termos)
    mascara = df['texto_completo'].str.contains(padrao, case=False, na=False, regex=True)
    return df[mascara].groupby(df['data_dt'].dt.to_period(freq)).size()

# Definição dos grupos para comparação
lista_tech_reg = ['inteligência artificial', ' IA ', 'digital', 'tecnologia', 'governança', 'regulação', 'padronização']
lista_china = ['China']

# Gerar as séries temporais (Trimestral para melhor visualização de tendência)
serie_tech = obter_serie_bloco(lista_tech_reg, freq='Q')
serie_china = obter_serie_bloco(lista_china, freq='Q')

# Criar o gráfico com dois eixos Y
fig, ax1 = plt.subplots(figsize=(12, 6))

# Eixo 1: Tecnologia e Regulação
color_tech = 'tab:blue'
ax1.set_xlabel('Trimestre')
ax1.set_ylabel('Notas: Tecnologia & Regulação', color=color_tech, fontsize=12, fontweight='bold')
lns1 = ax1.plot(serie_tech.index.astype(str), serie_tech.values, color=color_tech, marker='o', linewidth=3, label='Tecnologia & Regulação')
ax1.tick_params(axis='y', labelcolor=color_tech)

# Eixo 2: China (Eixo Gêmeo)
ax2 = ax1.twinx()
color_china = 'tab:red'
ax2.set_ylabel('Notas: China', color=color_china, fontsize=12, fontweight='bold')
lns2 = ax2.plot(serie_china.index.astype(str), serie_china.values, color=color_china, marker='s', linewidth=3, linestyle='--', label='China')
ax2.tick_params(axis='y', labelcolor=color_china)

# Títulos e Legendas
plt.title('Correlação Temporal: Agenda Digital/Regulatória vs. Menções à China', fontsize=14, pad=20)
ax1.grid(True, linestyle='--', alpha=0.6)

# Juntar as legendas de ambos os eixos
lns = lns1 + lns2
labs = [l.get_label() for l in lns]
ax1.legend(lns, labs, loc='upper left')

fig.tight_layout()
plt.xticks(rotation=45)
plt.show()

# Cálculo simples de correlação de Pearson para confirmar
df_corr = pd.DataFrame({'tech': serie_tech, 'china': serie_china}).fillna(0)
correlacao = df_corr.corr().iloc[0,1]
print(f"Índice de correlação de Pearson entre os dois grupos: {correlacao:.2f}")

In [ ]:
def obter_serie_bloco(lista_termos, freq='M'):
    """Retorna a contagem de termos agrupada por tempo (Mensal por padrão)."""
    padrao = '|'.join(lista_termos)
    mascara = df['texto_completo'].str.contains(padrao, case=False, na=False, regex=True)
    return df[mascara].groupby(df['data_dt'].dt.to_period(freq)).size()

# Definição dos grupos para comparação
lista_tech_reg = ['inteligência artificial', ' IA ', 'digital', 'tecnologia', 'governança', 'regulação', 'padronização']
lista_china = ['China']

# Gerar as séries temporais Mensais
serie_tech = obter_serie_bloco(lista_tech_reg, freq='M')
serie_china = obter_serie_bloco(lista_china, freq='M')

# Criar o gráfico com dois eixos Y
fig, ax1 = plt.subplots(figsize=(14, 6))

# Eixo 1: Tecnologia e Regulação
color_tech = 'tab:blue'
ax1.set_xlabel('Mês de Publicação')
ax1.set_ylabel('Notas: Tecnologia & Regulação', color=color_tech, fontsize=12, fontweight='bold')
lns1 = ax1.plot(serie_tech.index.astype(str), serie_tech.values, color=color_tech, marker='.', linewidth=2, label='Tecnologia & Regulação')
ax1.tick_params(axis='y', labelcolor=color_tech)

# Eixo 2: China (Eixo Gêmeo)
ax2 = ax1.twinx()
color_china = 'tab:red'
ax2.set_ylabel('Notas: China', color=color_china, fontsize=12, fontweight='bold')
lns2 = ax2.plot(serie_china.index.astype(str), serie_china.values, color=color_china, marker='.', linewidth=2, linestyle='--', label='China')
ax2.tick_params(axis='y', labelcolor=color_china)

# Títulos e Legendas
plt.title('Evolução Mensal: Agenda Digital/Regulatória vs. Menções à China', fontsize=14, pad=20)
ax1.grid(True, linestyle='--', alpha=0.4)

# Juntar as legendas
lns = lns1 + lns2
labs = [l.get_label() for l in lns]
ax1.legend(lns, labs, loc='upper left')

# Ajustar visualização das datas no eixo X para não amontoar
plt.xticks(rotation=90)
fig.tight_layout()
plt.show()

# Recalcular correlação mensal
df_corr_mensal = pd.DataFrame({'tech': serie_tech, 'china': serie_china}).fillna(0)
correlacao = df_corr_mensal.corr().iloc[0,1]
print(f"Índice de correlação de Pearson (Mensal): {correlacao:.2f}")

In [ ]:
def obter_serie_bloco(lista_termos, freq='6ME'):
    """Retorna a contagem de termos agrupada por tempo (Semestral)."""
    padrao = '|'.join(lista_termos)
    mascara = df['texto_completo'].str.contains(padrao, case=False, na=False, regex=True)
    # Usando pd.Grouper para garantir o agrupamento por 6 meses (semestre)
    return df[mascara].groupby(pd.Grouper(key='data_dt', freq=freq)).size()

# Definição dos grupos
lista_tech_reg = ['inteligência artificial', ' IA ', 'digital', 'tecnologia', 'governança', 'regulação', 'padronização']
lista_china = ['China']

# Gerar as séries temporais Semestrais
serie_tech = obter_serie_bloco(lista_tech_reg)
serie_china = obter_serie_bloco(lista_china)

# Criar o gráfico com dois eixos Y
fig, ax1 = plt.subplots(figsize=(12, 6))

# Formatação das datas para o eixo X (Ano e semestre aproximado)
eixo_x = [f"{d.year} - S{1 if d.month <= 6 else 2}" for d in serie_tech.index]

# Eixo 1: Tecnologia e Regulação
color_tech = 'tab:blue'
ax1.set_xlabel('Semestre')
ax1.set_ylabel('Notas: Tecnologia & Regulação', color=color_tech, fontsize=12, fontweight='bold')
lns1 = ax1.plot(eixo_x, serie_tech.values, color=color_tech, marker='o', linewidth=3, label='Tecnologia & Regulação')
ax1.tick_params(axis='y', labelcolor=color_tech)

# Eixo 2: China (Eixo Gêmeo)
ax2 = ax1.twinx()
color_china = 'tab:red'
ax2.set_ylabel('Notas: China', color=color_china, fontsize=12, fontweight='bold')
lns2 = ax2.plot(eixo_x, serie_china.values, color=color_china, marker='s', linewidth=3, linestyle='--', label='China')
ax2.tick_params(axis='y', labelcolor=color_china)

# Títulos e Legendas
plt.title('Evolução Semestral: Agenda Digital/Regulatória vs. Menções à China', fontsize=14, pad=20)
ax1.grid(True, linestyle='--', alpha=0.5)

lns = lns1 + lns2
labs = [l.get_label() for l in lns]
ax1.legend(lns, labs, loc='upper left')

fig.tight_layout()
plt.show()

# Recalcular correlação semestral
df_corr_sem = pd.DataFrame({'tech': serie_tech, 'china': serie_china}).fillna(0)
correlacao = df_corr_sem.corr().iloc[0,1]
print(f"Índice de correlação de Pearson (Semestral): {correlacao:.2f}")

In [ ]:
def obter_serie_bloco(lista_termos, freq='6ME'):
    """Retorna a contagem de termos agrupada por tempo (Semestral)."""
    padrao = '|'.join(lista_termos)
    mascara = df['texto_completo'].str.contains(padrao, case=False, na=False, regex=True)
    return df[mascara].groupby(pd.Grouper(key='data_dt', freq=freq)).size()

# Definição dos grupos
lista_tech_reg = ['inteligência artificial', ' IA ', 'digital', 'tecnologia', 'governança', 'regulação', 'padronização']
lista_china = ['China']

# Gerar as séries temporais Semestrais
serie_tech = obter_serie_bloco(lista_tech_reg)
serie_china = obter_serie_bloco(lista_china)

# Criar o gráfico com dois eixos Y
fig, ax1 = plt.subplots(figsize=(14, 7))

# Formatação das datas para o eixo X
eixo_x_labels = [f"{d.year} - S{1 if d.month <= 6 else 2}" for d in serie_tech.index]
eixo_x_indices = range(len(eixo_x_labels))

# Eixo 1: Tecnologia e Regulação
color_tech = 'tab:blue'
ax1.set_xlabel('Semestre')
ax1.set_ylabel('Notas: Tecnologia & Regulação', color=color_tech, fontsize=12, fontweight='bold')
lns1 = ax1.plot(eixo_x_indices, serie_tech.values, color=color_tech, marker='o', linewidth=3, label='Tecnologia & Regulação')
ax1.tick_params(axis='y', labelcolor=color_tech)

# Eixo 2: China (Eixo Gêmeo)
ax2 = ax1.twinx()
color_china = 'tab:red'
ax2.set_ylabel('Notas: China', color=color_china, fontsize=12, fontweight='bold')
lns2 = ax2.plot(eixo_x_indices, serie_china.values, color=color_china, marker='s', linewidth=3, linestyle='--', label='China')
ax2.tick_params(axis='y', labelcolor=color_china)

# Adicionar Eventos diplomáticos-chave
# Como o eixo X é categórico, precisamos calcular a posição aproximada
for data_evento, rotulo in [
    ('2023-03-12', '12-15 Abr/2023\n3ª Visita Lula à China'),
    ('2023-08-24', '24 Ago/2023\nReunião COSBAN'),
    ('2024-06-04', '4-7 Jun/2024\nVII Plenária COSBAN'),
    ('2024-11-05', '19-20 Nov/2024\nVisita Xi Jinping'),
    ('2025-04-01', 'Mai/2025\n4ª Visita Lula + BRICS IA'),
]:
    ts = pd.Timestamp(data_evento)
    sem_label = f"{ts.year} - S{1 if ts.month <= 6 else 2}"
    if sem_label in eixo_x_labels:
        idx = eixo_x_labels.index(sem_label)
        ax1.axvline(idx, color='gray', linestyle=':', linewidth=1.5, alpha=0.7)
        ax1.text(idx, ax1.get_ylim()[1]*0.95, rotulo, rotation=90, va='top', ha='right', fontsize=9, color='black', fontweight='bold')

# Títulos e Legendas
plt.title('Evolução Semestral e Eventos-Chave: Tecnologia/Regulação vs. China', fontsize=14, pad=25)
ax1.set_xticks(eixo_x_indices)
ax1.set_xticklabels(eixo_x_labels)
ax1.grid(True, linestyle='--', alpha=0.3)

lns = lns1 + lns2
labs = [l.get_label() for l in lns]
ax1.legend(lns, labs, loc='upper left')

fig.tight_layout()
plt.show()

# Correlação
df_corr_sem = pd.DataFrame({'tech': serie_tech, 'china': serie_china}).fillna(0)
print(f"Índice de correlação de Pearson (Semestral): {df_corr_sem.corr().iloc[0,1]:.2f}")

In [ ]:
def obter_serie_bloco(lista_termos, freq='6ME'):
    """Retorna a contagem de termos agrupada por tempo (Semestral)."""
    padrao = '|'.join(lista_termos)
    mascara = df['texto_completo'].str.contains(padrao, case=False, na=False, regex=True)
    return df[mascara].groupby(pd.Grouper(key='data_dt', freq=freq)).size()

# Definição dos grupos
lista_tech_reg = ['inteligência artificial', ' IA ', 'digital', 'tecnologia', 'governança', 'regulação', 'padronização']
lista_china = ['China']

# Gerar as séries temporais Semestrais
serie_tech = obter_serie_bloco(lista_tech_reg)
serie_china = obter_serie_bloco(lista_china)

# Criar o gráfico
fig, ax1 = plt.subplots(figsize=(15, 8))

# Plotamos usando o índice de tempo real (data_dt) para que o eixo X seja contínuo e preciso
lns1 = ax1.plot(serie_tech.index, serie_tech.values, color='tab:blue', marker='o', linewidth=3, label='Tecnologia & Regulação')
ax1.set_ylabel('Notas: Tecnologia & Regulação', color='tab:blue', fontsize=12, fontweight='bold')
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
lns2 = ax2.plot(serie_china.index, serie_china.values, color='tab:red', marker='s', linewidth=3, linestyle='--', label='China')
ax2.set_ylabel('Notas: China', color='tab:red', fontsize=12, fontweight='bold')
ax2.tick_params(axis='y', labelcolor='tab:red')

# Adicionar Eventos diplomáticos na posição EXATA do tempo
eventos = [
    ('2023-04-12', '12-15 Abr/2023\n3ª Visita Lula à China'),
    ('2023-08-24', '24 Ago/2023\nReunião COSBAN'),
    ('2024-06-04', '4-7 Jun/2024\nVII Plenária COSBAN'),
    ('2024-11-19', '19-20 Nov/2024\nVisita Xi Jinping'),
    ('2025-05-10', 'Mai/2025\n4ª Visita Lula + BRICS IA'),
]

for data_evento, rotulo in eventos:
    ts = pd.Timestamp(data_evento)
    if ts >= serie_tech.index.min() and ts <= serie_tech.index.max():
        ax1.axvline(ts, color='gray', linestyle=':', linewidth=1.5, alpha=0.8)
        # Posiciona o texto ligeiramente à frente da linha para melhor leitura
        ax1.text(ts, ax1.get_ylim()[1]*0.98, rotulo, rotation=90, va='top', ha='right', fontsize=9, color='black', fontweight='bold')

# Formatação do Eixo X para mostrar os Semestres como rótulos
ax1.set_xlabel('Linha do Tempo (Agregação Semestral)')
eixo_x_ticks = serie_tech.index
eixo_x_labels = [f"{d.year} - S{1 if d.month <= 6 else 2}" for d in eixo_x_ticks]
ax1.set_xticks(eixo_x_ticks)
ax1.set_xticklabels(eixo_x_labels, rotation=45)

plt.title('Evolução Semestral: Posição Precisa de Eventos Diplomáticos vs. Temas de Interesse', fontsize=14, pad=30)
ax1.grid(True, linestyle='--', alpha=0.3)

lns = lns1 + lns2
labs = [l.get_label() for l in lns]
ax1.legend(lns, labs, loc='upper left')

fig.tight_layout()
plt.show()

In [ ]:
def obter_serie_bloco(lista_termos, freq='4ME'):
    """Retorna a contagem de termos agrupada por quadrimestre (4 meses)."""
    padrao = '|'.join(lista_termos)
    mascara = df['texto_completo'].str.contains(padrao, case=False, na=False, regex=True)
    return df[mascara].groupby(pd.Grouper(key='data_dt', freq=freq)).size()

# Definição dos grupos
lista_tech_reg = ['inteligência artificial', ' IA ', 'digital', 'tecnologia', 'governança', 'regulação', 'padronização']
lista_china = ['China']

# Gerar as séries temporais Quadrimestrais
serie_tech = obter_serie_bloco(lista_tech_reg)
serie_china = obter_serie_bloco(lista_china)

# Criar o gráfico com dois eixos Y
fig, ax1 = plt.subplots(figsize=(12, 6))

# Formatação das datas para o eixo X (Ano e mês final do quadrimestre)
eixo_x = [d.strftime('%Y-%m') for d in serie_tech.index]

# Eixo 1: Tecnologia e Regulação
color_tech = 'tab:blue'
ax1.set_xlabel('Quadrimestre (Final do período)')
ax1.set_ylabel('Notas: Tecnologia & Regulação', color=color_tech, fontsize=12, fontweight='bold')
lns1 = ax1.plot(eixo_x, serie_tech.values, color=color_tech, marker='o', linewidth=3, label='Tecnologia & Regulação')
ax1.tick_params(axis='y', labelcolor=color_tech)

# Eixo 2: China (Eixo Gêmeo)
ax2 = ax1.twinx()
color_china = 'tab:red'
ax2.set_ylabel('Notas: China', color=color_china, fontsize=12, fontweight='bold')
lns2 = ax2.plot(eixo_x, serie_china.values, color=color_china, marker='s', linewidth=3, linestyle='--', label='China')
ax2.tick_params(axis='y', labelcolor=color_china)

# Títulos e Legendas
plt.title('Evolução Quadrimestral: Agenda Digital/Regulatória vs. Menções à China', fontsize=14, pad=20)
ax1.grid(True, linestyle='--', alpha=0.5)

lns = lns1 + lns2
labs = [l.get_label() for l in lns]
ax1.legend(lns, labs, loc='upper left')

fig.tight_layout()
plt.show()

# Recalcular correlação quadrimestral
df_corr_quad = pd.DataFrame({'tech': serie_tech, 'china': serie_china}).fillna(0)
correlacao = df_corr_quad.corr().iloc[0,1]
print(f"Índice de correlação de Pearson (Quadrimestral): {correlacao:.2f}")

In [ ]:
def obter_serie_bloco(lista_termos, freq='4ME'):
    """Retorna a contagem de termos agrupada por quadrimestre (4 meses)."""
    padrao = '|'.join(lista_termos)
    mascara = df['texto_completo'].str.contains(padrao, case=False, na=False, regex=True)
    return df[mascara].groupby(pd.Grouper(key='data_dt', freq=freq)).size()

# Definição dos grupos
lista_tech_reg = ['inteligência artificial', ' IA ', 'digital', 'tecnologia', 'governança', 'regulação', 'padronização']
lista_china = ['China']

# Gerar as séries temporais Quadrimestrais
serie_tech = obter_serie_bloco(lista_tech_reg)
serie_china = obter_serie_bloco(lista_china)

# Criar o gráfico com dois eixos Y
fig, ax1 = plt.subplots(figsize=(12, 6))

# Formatação das datas para o eixo X (Ano e mês final do quadrimestre)
eixo_x = [d.strftime('%Y-%m') for d in serie_tech.index]

# Eixo 1: Tecnologia e Regulação
color_tech = 'tab:blue'
ax1.set_xlabel('Quadrimestre (Final do período)')
ax1.set_ylabel('Notas: Tecnologia & Regulação', color=color_tech, fontsize=12, fontweight='bold')
lns1 = ax1.plot(eixo_x, serie_tech.values, color=color_tech, marker='o', linewidth=3, label='Tecnologia & Regulação')
ax1.tick_params(axis='y', labelcolor=color_tech)

# Eixo 2: China (Eixo Gêmeo)
ax2 = ax1.twinx()
color_china = 'tab:red'
ax2.set_ylabel('Notas: China', color=color_china, fontsize=12, fontweight='bold')
lns2 = ax2.plot(eixo_x, serie_china.values, color=color_china, marker='s', linewidth=3, linestyle='--', label='China')
ax2.tick_params(axis='y', labelcolor=color_china)

# Títulos e Legendas
plt.title('Evolução Quadrimestral: Agenda Digital/Regulatória vs. Menções à China', fontsize=14, pad=20)
ax1.grid(True, linestyle='--', alpha=0.5)

lns = lns1 + lns2
labs = [l.get_label() for l in lns]
ax1.legend(lns, labs, loc='upper left')

fig.tight_layout()
plt.show()

# Recalcular correlação quadrimestral
df_corr_quad = pd.DataFrame({'tech': serie_tech, 'china': serie_china}).fillna(0)
correlacao = df_corr_quad.corr().iloc[0,1]
print(f"Índice de correlação de Pearson (Quadrimestral): {correlacao:.2f}")

In [ ]:
def evolucao_bloco_termos_bimestral(lista_termos, onde='completo'):
    """Soma as menções de um grupo de termos por bimestre."""
    coluna = {'titulo': 'titulo', 'texto': 'texto', 'completo': 'texto_completo'}[onde]

    # Cria uma máscara que é verdadeira se QUALQUER um dos termos estiver presente
    padrao = '|'.join(lista_termos)
    mascara = df[coluna].str.contains(padrao, case=False, na=False, regex=True)

    # Agrupa por períodos de 2 meses (bimestre) usando a data
    # Usamos o Grouper para manter o índice como datetime para facilitar a plotagem de eventos
    return df[mascara].groupby(pd.Grouper(key='data_dt', freq='2ME')).size()

# Definição dos blocos de termos
blocos = {
    'TERMOS TECNOLÓGICOS': ['inteligência artificial', ' IA ', 'digital', 'infraestrutura digital', 'tecnologia'],
    'TERMOS REGULATÓRIOS': ['governança', 'padronização', 'regulação', 'cooperação', 'sul-sul'],
    'TERMOS DE PAÍSES': ['China', 'BRICS']
}

fig, ax = plt.subplots(figsize=(15, 7))

for nome_bloco, lista in blocos.items():
    serie = evolucao_bloco_termos_bimestral(lista)
    if len(serie) > 0:
        # Plotamos usando o índice de tempo real
        ax.plot(serie.index, serie.values, marker='o', label=nome_bloco, linewidth=2)

# Adicionar Eventos diplomáticos na posição EXATA do tempo
eventos = [
    # 2023
    ('2023-04-12', '12-15 Abr/2023\n3ª Visita Lula à China'),
    ('2023-08-24', '24 Ago/2023\nReunião COSBAN'),

    # 2024
    ('2024-06-04', '4-7 Jun/2024\nVII Plenária COSBAN'),
    ('2024-11-19', '19-20 Nov/2024\nVisita Xi Jinping'),

    # 2025
    ('2025-05-10', '10-13 Mai/2025\n4ª Visita Lula à China'),
    # ('2025-05-20', '20 Mai/2025\nFórum de Alto Nível do BRICS sobre IA'),
    # ('2025-05-30', '30 Mai/2025\nFórum Digital BRICS'),
    ('2025-07-06', '6-7 Jul/2025\n17ª Cúpula do BRICS'),
    # ('2025-08-04', '4 Ago/2025\nAcordo Serpro × CAICT'),
    # ('2025-09-16', '16 Set/2025\n11ª Reunião Subcomissão COSBAN'),

    # 2026
    ('2026-04-10', '10 Abr/2026\nAcordo MCTI + Serpro × iFlytek')
]

for data_evento, rotulo in eventos:
    ts = pd.Timestamp(data_evento)
    ax.axvline(ts, color='gray', linestyle=':', linewidth=1.5, alpha=0.8)
    # Posiciona o texto
    ax.text(ts, ax.get_ylim()[1]*0.95, rotulo, rotation=90, va='top', ha='right', fontsize=9, color='black', fontweight='bold')

# Formatação do Eixo X
ax.set_title('Evolução Bimestral por Blocos Temáticos e Eventos Diplomáticos', fontsize=14, pad=20)
ax.set_xlabel('Linha do Tempo (Bimestres)')
ax.set_ylabel('Número de notas')

# Ajustar os ticks do eixo X para mostrar os bimestres de forma legível
eixo_x_ticks = df.groupby(pd.Grouper(key='data_dt', freq='2ME')).size().index
ax.set_xticks(eixo_x_ticks)
ax.set_xticklabels([d.strftime('%Y-%m') for d in eixo_x_ticks], rotation=45, ha='right')

ax.legend()
ax.grid(linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def evolucao_bloco_termos_bimestral(lista_termos, onde='completo', normalizado=True, usar_word_boundary=False):
    """
    Soma as menções de um grupo de termos por bimestre.

    normalizado=True -> retorna % de notas do bimestre que contêm o termo,
    em vez de contagem bruta. Isso isola MUDANÇA DE PRIORIDADE RELATIVA
    de simples crescimento no volume total de notas do Itamaraty.

    usar_word_boundary=True -> adiciona \b nos termos curtos (EUA, UE, IA)
    para evitar falsos positivos (ex: "UE" dentro de "sequência").
    """
    coluna = {'titulo': 'titulo', 'texto': 'texto', 'completo': 'texto_completo'}[onde]

    if usar_word_boundary:
        termos_escapados = [rf'\b{t.strip()}\b' if len(t.strip()) <= 4 else t for t in lista_termos]
    else:
        termos_escapados = lista_termos

    padrao = '|'.join(termos_escapados)
    mascara = df[coluna].str.contains(padrao, case=False, na=False, regex=True)

    contagem = df[mascara].groupby(pd.Grouper(key='data_dt', freq='2ME')).size()

    if not normalizado:
        return contagem

    total_periodo = df.groupby(pd.Grouper(key='data_dt', freq='2ME')).size()
    pct = (contagem / total_periodo * 100).fillna(0)
    return pct


# --- Blocos de termos ---
# ATENÇÃO: revise a alocação de BRICS aqui. Deixei junto de China por ser
# o bloco multilateral onde a China exerce liderança normativa, mas se
# quiser tratar BRICS como categoria à parte (nem China nem Ocidente),
# é só criar um quarto bloco.
blocos = {
    'TERMOS TECNOLÓGICOS': ['inteligência artificial', ' IA ', 'digital', 'infraestrutura digital', 'tecnologia'],
    'TERMOS REGULATÓRIOS': ['governança', 'padronização', 'regulação', 'cooperação', 'sul-sul'],
    'CHINA / BRICS': ['China', 'chinês', 'chinesa', 'BRICS'],
    'UE': ['União Europeia', ' UE ', 'Europa', 'europeu', 'europeia'],
    'EUA': ['Estados Unidos', ' EUA ']
}

NORMALIZADO = True          # True = % do total de notas do bimestre | False = contagem bruta
MOSTRAR_MEDIA_MOVEL = True  # sobrepõe média móvel de 2 períodos em traço mais grosso

fig, ax = plt.subplots(figsize=(16, 8))

cores = plt.cm.tab10.colors  # paleta consistente entre linha bruta e média móvel

for i, (nome_bloco, lista) in enumerate(blocos.items()):
    serie = evolucao_bloco_termos_bimestral(lista, normalizado=NORMALIZADO, usar_word_boundary=True)
    if len(serie) == 0:
        continue

    cor = cores[i % len(cores)]

    if MOSTRAR_MEDIA_MOVEL:
        # linha bruta mais fina e clara (mantém o dado real visível)
        ax.plot(serie.index, serie.values, marker='o', markersize=4,
                linewidth=1, alpha=0.35, color=cor)
        # média móvel em destaque (mostra a tendência de fundo)
        media_movel = serie.rolling(window=2, min_periods=1).mean()
        ax.plot(serie.index, media_movel.values, linewidth=2.5,
                color=cor, label=nome_bloco)
    else:
        ax.plot(serie.index, serie.values, marker='o', linewidth=2,
                color=cor, label=nome_bloco)

# --- Eventos diplomáticos ---
eventos = [
    ('2023-04-12', '12-15 Abr/2023\n3ª Visita Lula à China'),
    ('2023-08-24', '24 Ago/2023\nReunião COSBAN'),
    ('2024-06-04', '4-7 Jun/2024\nVII Plenária COSBAN'),
    ('2024-11-19', '19-20 Nov/2024\nVisita Xi Jinping'),
    ('2025-05-10', '10-13 Mai/2025\n4ª Visita Lula à China'),
    ('2025-07-06', '6-7 Jul/2025\n17ª Cúpula do BRICS'),
    ('2026-04-10', '10 Abr/2026\nAcordo MCTI + Serpro × iFlytek'),
]

ylim_atual = ax.get_ylim()

for idx, (data_evento, rotulo) in enumerate(eventos):
    ts = pd.Timestamp(data_evento)

    # linha vertical do evento
    ax.axvline(ts, color='gray', linestyle=':', linewidth=1.5, alpha=0.8)

    # NOVO: sombreamento da janela de possível efeito defasado (2 meses = 1 bimestre)
    ax.axvspan(ts, ts + pd.DateOffset(months=2), color='gray', alpha=0.06, zorder=0)

    # NOVO: altura alternada para reduzir sobreposição de rótulos
    altura = ylim_atual[1] * (0.95 if idx % 2 == 0 else 0.75)
    ax.text(ts, altura, rotulo, rotation=90, va='top', ha='right',
            fontsize=8.5, color='black', fontweight='bold')

# --- Formatação ---
titulo_sufixo = '(% de notas do período)' if NORMALIZADO else '(contagem absoluta)'
ax.set_title(f'Evolução Bimestral por Blocos Temáticos e Eventos Diplomáticos {titulo_sufixo}',
             fontsize=14, pad=20)
ax.set_xlabel('Linha do Tempo (Bimestres)')
ax.set_ylabel('% de notas do bimestre' if NORMALIZADO else 'Número de notas')

eixo_x_ticks = df.groupby(pd.Grouper(key='data_dt', freq='2ME')).size().index
ax.set_xticks(eixo_x_ticks)
ax.set_xticklabels([d.strftime('%Y-%m') for d in eixo_x_ticks], rotation=45, ha='right')

ax.legend(loc='upper left', framealpha=0.9)
ax.grid(linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def evolucao_bloco_termos_bimestral(lista_termos, onde='completo', normalizado=True, usar_word_boundary=False):
    """
    Soma as menções de um grupo de termos por bimestre.

    normalizado=True -> retorna % de notas do bimestre que contêm o termo,
    em vez de contagem bruta. Isso isola MUDANÇA DE PRIORIDADE RELATIVA
    de simples crescimento no volume total de notas do Itamaraty.

    usar_word_boundary=True -> adiciona \b nos termos curtos (EUA, UE, IA)
    para evitar falsos positivos (ex: "UE" dentro de "sequência").
    """
    coluna = {'titulo': 'titulo', 'texto': 'texto', 'completo': 'texto_completo'}[onde]

    if usar_word_boundary:
        termos_escapados = [rf'\b{t.strip()}\b' if len(t.strip()) <= 4 else t for t in lista_termos]
    else:
        termos_escapados = lista_termos

    padrao = '|'.join(termos_escapados)
    mascara = df[coluna].str.contains(padrao, case=False, na=False, regex=True)

    contagem = df[mascara].groupby(pd.Grouper(key='data_dt', freq='2ME')).size()

    if not normalizado:
        return contagem

    total_periodo = df.groupby(pd.Grouper(key='data_dt', freq='2ME')).size()
    pct = (contagem / total_periodo * 100).fillna(0)
    return pct


# --- Blocos de termos ---
# ATENÇÃO: revise a alocação de BRICS aqui. Deixei junto de China por ser
# o bloco multilateral onde a China exerce liderança normativa, mas se
# quiser tratar BRICS como categoria à parte (nem China nem Ocidente),
# é só criar um quarto bloco.
blocos = {
    'TERMOS TECNOLÓGICOS': ['inteligência artificial', ' IA ', 'digital', 'infraestrutura digital', 'tecnologia'],
    'TERMOS REGULATÓRIOS': ['governança', 'padronização', 'regulação', 'cooperação', 'sul-sul'],
    'CHINA / BRICS': ['China', 'chinês', 'chinesa', 'BRICS'],
    'OCIDENTE (EUROPA + EUA)': ['Estados Unidos', ' EUA ', 'União Europeia', ' UE ', 'Europa', 'europeu', 'europeia'],
}

NORMALIZADO = True          # True = % do total de notas do bimestre | False = contagem bruta
MOSTRAR_MEDIA_MOVEL = False  # sobrepõe média móvel de 2 períodos em traço mais grosso

fig, ax = plt.subplots(figsize=(16, 8))

cores = plt.cm.tab10.colors  # paleta consistente entre linha bruta e média móvel

for i, (nome_bloco, lista) in enumerate(blocos.items()):
    serie = evolucao_bloco_termos_bimestral(lista, normalizado=NORMALIZADO, usar_word_boundary=True)
    if len(serie) == 0:
        continue

    cor = cores[i % len(cores)]

    if MOSTRAR_MEDIA_MOVEL:
        # linha bruta mais fina e clara (mantém o dado real visível)
        ax.plot(serie.index, serie.values, marker='o', markersize=4,
                linewidth=1, alpha=0.35, color=cor)
        # média móvel em destaque (mostra a tendência de fundo)
        media_movel = serie.rolling(window=2, min_periods=1).mean()
        ax.plot(serie.index, media_movel.values, linewidth=2.5,
                color=cor, label=nome_bloco)
    else:
        ax.plot(serie.index, serie.values, marker='o', linewidth=2,
                color=cor, label=nome_bloco)

# --- Eventos diplomáticos ---
eventos = [
    ('2023-04-12', '12-15 Abr/2023\n3ª Visita Lula à China'),
    ('2023-08-24', '24 Ago/2023\nReunião COSBAN'),
    ('2024-06-04', '4-7 Jun/2024\nVII Plenária COSBAN'),
    ('2024-11-19', '19-20 Nov/2024\nVisita Xi Jinping'),
    ('2025-05-10', '10-13 Mai/2025\n4ª Visita Lula à China'),
    ('2025-07-06', '6-7 Jul/2025\n17ª Cúpula do BRICS'),
    ('2026-04-10', '10 Abr/2026\nAcordo MCTI + Serpro × iFlytek'),
]

ylim_atual = ax.get_ylim()

for idx, (data_evento, rotulo) in enumerate(eventos):
    ts = pd.Timestamp(data_evento)

    # linha vertical do evento
    ax.axvline(ts, color='gray', linestyle=':', linewidth=1.5, alpha=0.8)

    # NOVO: sombreamento da janela de possível efeito defasado (2 meses = 1 bimestre)
    ax.axvspan(ts, ts + pd.DateOffset(months=2), color='gray', alpha=0.06, zorder=0)

    # NOVO: altura alternada para reduzir sobreposição de rótulos
    altura = ylim_atual[1] * (0.95 if idx % 2 == 0 else 0.75)
    ax.text(ts, altura, rotulo, rotation=90, va='top', ha='right',
            fontsize=8.5, color='black', fontweight='bold')

# --- Formatação ---
titulo_sufixo = '(% de notas do período)' if NORMALIZADO else '(contagem absoluta)'
ax.set_title(f'Evolução Bimestral por Blocos Temáticos e Eventos Diplomáticos {titulo_sufixo}',
             fontsize=14, pad=20)
ax.set_xlabel('Linha do Tempo (Bimestres)')
ax.set_ylabel('% de notas do bimestre' if NORMALIZADO else 'Número de notas')

eixo_x_ticks = df.groupby(pd.Grouper(key='data_dt', freq='2ME')).size().index
ax.set_xticks(eixo_x_ticks)
ax.set_xticklabels([d.strftime('%Y-%m') for d in eixo_x_ticks], rotation=45, ha='right')

ax.legend(loc='upper left', framealpha=0.9)
ax.grid(linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def evolucao_bloco_termos_bimestral(lista_termos, onde='completo', normalizado=True, usar_word_boundary=False):
    """
    Soma as menções de um grupo de termos por bimestre.

    normalizado=True -> retorna % de notas do bimestre que contêm o termo,
    em vez de contagem bruta. Isso isola MUDANÇA DE PRIORIDADE RELATIVA
    de simples crescimento no volume total de notas do Itamaraty.

    usar_word_boundary=True -> adiciona \b nos termos curtos (EUA, UE, IA)
    para evitar falsos positivos (ex: "UE" dentro de "sequência").
    """
    coluna = {'titulo': 'titulo', 'texto': 'texto', 'completo': 'texto_completo'}[onde]

    if usar_word_boundary:
        termos_escapados = [rf'\b{t.strip()}\b' if len(t.strip()) <= 4 else t for t in lista_termos]
    else:
        termos_escapados = lista_termos

    padrao = '|'.join(termos_escapados)
    mascara = df[coluna].str.contains(padrao, case=False, na=False, regex=True)

    contagem = df[mascara].groupby(pd.Grouper(key='data_dt', freq='2ME')).size()

    if not normalizado:
        return contagem

    total_periodo = df.groupby(pd.Grouper(key='data_dt', freq='2ME')).size()
    pct = (contagem / total_periodo * 100).fillna(0)
    return pct


# --- Blocos de termos ---
# ATENÇÃO: revise a alocação de BRICS aqui. Deixei junto de China por ser
# o bloco multilateral onde a China exerce liderança normativa, mas se
# quiser tratar BRICS como categoria à parte (nem China nem Ocidente),
# é só criar um quarto bloco.

# ============================================================
# BLOCOS DE TERMOS — versão expandida
# ============================================================
# Termos curtos (<=4 caracteres, ex: "IA", "UE", "5G") dependem do
# usar_word_boundary=True na função evolucao_bloco_termos_bimestral
# para não capturar substrings de outras palavras.

blocos = {

    # --------------------------------------------------------
    # A. TERMOS TECNOLÓGICOS GERAIS (neutros)
    # --------------------------------------------------------
    'TERMOS TECNOLÓGICOS': [
        'inteligência artificial', 'artificial intelligence', ' IA ', ' AI ',
        'aprendizado de máquina', 'machine learning',
        'big data', 'dados massivos',
        'algoritmo', 'algorithm',
        'modelo de linguagem', 'large language model', ' LLM ',
        'computação em nuvem', 'cloud computing',
        'infraestrutura digital', 'digital infrastructure',
        ' 5G ', ' 6G ',
        'semicondutor', 'semiconductor', ' chip', # cobre chip/chips
        'centro de dados', 'data center',
        'internet das coisas', 'internet of things', ' IoT ',
        'cibersegurança', 'cybersecurity',
        'economia digital', 'digital economy',
        'transformação digital', 'digital transformation',
        'inovação tecnológica', 'technological innovation',
        'comércio digital', 'digital trade',
    ],

    # --------------------------------------------------------
    # B. GOVERNANÇA — léxico mais associado à CHINA
    # --------------------------------------------------------
    'GOVERNANÇA (CHINA)': [
        'soberania digital', 'digital sovereignty',
        'soberania de dados', 'data sovereignty',
        'controlável e seguro', 'secure and controllable',
        'autossuficiência tecnológica', 'technological self-reliance', 'self-sufficiency',
        'inovação indígena', 'indigenous innovation',
        'cooperação sul-sul', 'south-south cooperation',
        'multipolaridade', 'multipolarity',
        'ordem internacional justa', 'just international order',
        'comunidade de futuro compartilhado', 'community of shared future',
        'rota da seda digital', 'digital silk road',
        'cinturão e rota', 'belt and road', ' BRI ',
        'não-interferência', 'non-interference',
        'respeito mútuo', 'mutual respect',
        'global ai governance initiative',
        'governança global de ia orientada pelo desenvolvimento', 'development-oriented ai governance',
        'novo banco de desenvolvimento', 'new development bank',
    ],

    # --------------------------------------------------------
    # C. GOVERNANÇA — léxico mais associado ao OCIDENTE
    # --------------------------------------------------------
    'GOVERNANÇA (OCIDENTE)': [
        'ia centrada no ser humano', 'human-centric ai', 'human centered ai',
        'direitos humanos', 'human rights',
        'direitos fundamentais', 'fundamental rights',
        'dignidade humana', 'human dignity',
        'ia confiável', 'trustworthy ai',
        'abordagem baseada em risco', 'risk-based approach',
        'avaliação de impacto', 'impact assessment',
        'accountability', 'responsabilização',
        'transparência algorítmica', 'algorithmic transparency',
        'explicabilidade', 'explainability',
        'auditoria de ia', 'ai audit',
        'multistakeholder', 'múltiplas partes interessadas',
        'estado de direito', 'rule of law',
        'valores democráticos', 'democratic values',
        'proteção de dados', 'data protection',
        'privacidade', 'privacy',
        'ai act', 'blueprint for an ai bill of rights', 'nist ai rmf',
        'padrões técnicos internacionais', 'international technical standards',
    ],

    # --------------------------------------------------------
    # D. TERMOS CONTESTADOS (ambos os lados usam — não somar
    # diretamente aos blocos B/C sem checar contexto/frame)
    # --------------------------------------------------------
    'TERMOS CONTESTADOS': [
        'soberania', 'sovereignty',
        'cooperação internacional', 'international cooperation',
        'governança global de ia', 'global ai governance',
        'segurança', 'security',
        'desenvolvimento sustentável', 'sustainable development',
    ],

    # --------------------------------------------------------
    # ATORES / BLOCOS GEOPOLÍTICOS (já existentes no seu código)
    # --------------------------------------------------------
    'CHINA / BRICS': [
        'China', 'chinês', 'chinesa', 'chinese', 'BRICS',
    ],
    'OCIDENTE (EUA + UE)': [
        'Estados Unidos', ' EUA ', ' United States', ' USA ',
        'União Europeia', ' UE ', ' European Union', ' EU ',
        'Europa', 'europeu', 'europeia', 'European',
    ],
}

NORMALIZADO = True          # True = % do total de notas do bimestre | False = contagem bruta
MOSTRAR_MEDIA_MOVEL = False  # sobrepõe média móvel de 2 períodos em traço mais grosso

fig, ax = plt.subplots(figsize=(16, 8))

cores = plt.cm.tab10.colors  # paleta consistente entre linha bruta e média móvel

for i, (nome_bloco, lista) in enumerate(blocos.items()):
    serie = evolucao_bloco_termos_bimestral(lista, normalizado=NORMALIZADO, usar_word_boundary=True)
    if len(serie) == 0:
        continue

    cor = cores[i % len(cores)]

    if MOSTRAR_MEDIA_MOVEL:
        # linha bruta mais fina e clara (mantém o dado real visível)
        ax.plot(serie.index, serie.values, marker='o', markersize=4,
                linewidth=1, alpha=0.35, color=cor)
        # média móvel em destaque (mostra a tendência de fundo)
        media_movel = serie.rolling(window=2, min_periods=1).mean()
        ax.plot(serie.index, media_movel.values, linewidth=2.5,
                color=cor, label=nome_bloco)
    else:
        ax.plot(serie.index, serie.values, marker='o', linewidth=2,
                color=cor, label=nome_bloco)

# --- Eventos diplomáticos ---
eventos = [
    ('2023-04-12', '12-15 Abr/2023\n3ª Visita Lula à China'),
    ('2023-08-24', '24 Ago/2023\nReunião COSBAN'),
    ('2024-06-04', '4-7 Jun/2024\nVII Plenária COSBAN'),
    ('2024-11-19', '19-20 Nov/2024\nVisita Xi Jinping'),
    ('2025-05-10', '10-13 Mai/2025\n4ª Visita Lula à China'),
    ('2025-07-06', '6-7 Jul/2025\n17ª Cúpula do BRICS'),
    ('2026-04-10', '10 Abr/2026\nAcordo MCTI + Serpro × iFlytek'),
]

ylim_atual = ax.get_ylim()

for idx, (data_evento, rotulo) in enumerate(eventos):
    ts = pd.Timestamp(data_evento)

    # linha vertical do evento
    ax.axvline(ts, color='gray', linestyle=':', linewidth=1.5, alpha=0.8)

    # NOVO: sombreamento da janela de possível efeito defasado (2 meses = 1 bimestre)
    ax.axvspan(ts, ts + pd.DateOffset(months=2), color='gray', alpha=0.06, zorder=0)

    # NOVO: altura alternada para reduzir sobreposição de rótulos
    altura = ylim_atual[1] * (0.95 if idx % 2 == 0 else 0.75)
    ax.text(ts, altura, rotulo, rotation=90, va='top', ha='right',
            fontsize=8.5, color='black', fontweight='bold')

# --- Formatação ---
titulo_sufixo = '(% de notas do período)' if NORMALIZADO else '(contagem absoluta)'
ax.set_title(f'Evolução Bimestral por Blocos Temáticos e Eventos Diplomáticos {titulo_sufixo}',
             fontsize=14, pad=20)
ax.set_xlabel('Linha do Tempo (Bimestres)')
ax.set_ylabel('% de notas do bimestre' if NORMALIZADO else 'Número de notas')

eixo_x_ticks = df.groupby(pd.Grouper(key='data_dt', freq='2ME')).size().index
ax.set_xticks(eixo_x_ticks)
ax.set_xticklabels([d.strftime('%Y-%m') for d in eixo_x_ticks], rotation=45, ha='right')

ax.legend(loc='upper left', framealpha=0.9)
ax.grid(linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import pearsonr

# ============================================================
# 1. CÁLCULO DAS SÉRIES BASE
# ============================================================
tec = evolucao_bloco_termos_bimestral(blocos['TERMOS TECNOLÓGICOS'], normalizado=True, usar_word_boundary=True)
reg = evolucao_bloco_termos_bimestral(blocos['TERMOS REGULATÓRIOS'], normalizado=True, usar_word_boundary=True)
china = evolucao_bloco_termos_bimestral(blocos['CHINA / BRICS'], normalizado=True, usar_word_boundary=True)
# CORREÇÃO: A chave correta definida anteriormente é 'OCIDENTE (EUROPA + EUA)'
ocidente = evolucao_bloco_termos_bimestral(blocos['OCIDENTE (EUROPA + EUA)'], normalizado=True, usar_word_boundary=True)

# Gap = diferencial de atenção. Positivo = pende para China. Negativo = pende para Ocidente.
gap = (china - ocidente).reindex(tec.index).fillna(0)

# Médias móveis (suavização de 2 períodos) para destacar tendência sem esconder o dado bruto
tec_mm = tec.rolling(2, min_periods=1).mean()
reg_mm = reg.rolling(2, min_periods=1).mean()
gap_mm = gap.rolling(2, min_periods=1).mean()

# ============================================================
# 2. CORRELAÇÕES (evidência numérica, não só visual)
# ============================================================
r_reg, p_reg = pearsonr(gap.values, reg.reindex(gap.index).values)
r_tec, p_tec = pearsonr(gap.values, tec.reindex(gap.index).values)

def significancia(p):
    return "significativo" if p < 0.05 else "não significativo (nesta amostra)"

print(f"Correlação Gap(China-Ocidente) × Termos Regulatórios: r={r_reg:.2f}, p={p_reg:.3f} -> {significancia(p_reg)}")
print(f"Correlação Gap(China-Ocidente) × Termos Tecnológicos : r={r_tec:.2f}, p={p_tec:.3f} -> {significancia(p_tec)}")

# ============================================================
# 3. EVENTOS DIPLOMÁTICOS
# ============================================================
eventos = [
    ('2023-04-12', '3ª Visita Lula à China'),
    ('2023-08-24', 'Reunião COSBAN'),
    ('2024-06-04', 'VII Plenária COSBAN'),
    ('2024-11-19', 'Visita Xi Jinping'),
    ('2025-05-10', '4ª Visita Lula à China'),
    ('2025-07-06', '17ª Cúpula do BRICS'),
    ('2026-04-10', 'Acordo MCTI + Serpro × iFlytek'),
]

# ============================================================
# 4. FIGURA: DOIS PAINÉIS EMPILHADOS, EIXO X COMPARTILHADO
# ============================================================
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(16, 11), sharex=True,
    gridspec_kw={'height_ratios': [1, 1], 'hspace': 0.08}
)

# ---- PAINEL SUPERIOR: conteúdo (tecnologia + regulação) ----
ax1.plot(tec.index, tec.values, color='#1f77b4', linewidth=1, alpha=0.3)
ax1.plot(tec.index, tec_mm.values, color='#1f77b4', linewidth=2.5, label='Termos Tecnológicos')

ax1.plot(reg.index, reg.values, color='#ff7f0e', linewidth=1, alpha=0.3)
ax1.plot(reg.index, reg_mm.values, color='#ff7f0e', linewidth=2.5, label='Termos Regulatórios')

ax1.set_ylabel('% de notas do bimestre')
ax1.set_title('Relação entre Tecnologia, Governança e Alinhamento Geopolítico (MRE, 2023–2026)',
               fontsize=15, pad=15)
ax1.legend(loc='upper left', framealpha=0.9)
ax1.grid(linestyle='--', alpha=0.3)

# ---- PAINEL INFERIOR: direção geopolítica (gap China - Ocidente) ----
ax2.plot(gap.index, gap.values, color='black', linewidth=1, alpha=0.3)
ax2.plot(gap.index, gap_mm.values, color='black', linewidth=2.5, label='Gap China − Ocidente (suavizado)')
ax2.fill_between(gap.index, gap_mm.values, 0,
                  where=(gap_mm.values >= 0), color='green', alpha=0.25, interpolate=True,
                  label='Atenção pende para China')
ax2.fill_between(gap.index, gap_mm.values, 0,
                  where=(gap_mm.values < 0), color='red', alpha=0.25, interpolate=True,
                  label='Atenção pende para Ocidente')
ax2.axhline(0, color='gray', linewidth=1)

ax2.set_ylabel('Diferencial (p.p.)')
ax2.set_xlabel('Linha do Tempo (Bimestres)')
ax2.legend(loc='upper left', framealpha=0.9)
ax2.grid(linestyle='--', alpha=0.3)

# ---- EVENTOS: linhas verticais contínuas nos dois painéis ----
for idx, (data_evento, rotulo) in enumerate(eventos):
    ts = pd.Timestamp(data_evento)
    for ax in (ax1, ax2):
        ax.axvline(ts, color='gray', linestyle=':', linewidth=1.3, alpha=0.7)
        ax.axvspan(ts, ts + pd.DateOffset(months=2), color='gray', alpha=0.05, zorder=0)
    # rótulo só no painel de cima, altura alternada
    altura = ax1.get_ylim()[1] * (0.95 if idx % 2 == 0 else 0.75)
    ax1.text(ts, altura, rotulo, rotation=90, va='top', ha='right',
              fontsize=8, color='black', fontweight='bold')

# ---- Eixo X formatado ----
eixo_x_ticks = tec.index
ax2.set_xticks(eixo_x_ticks)
ax2.set_xticklabels([d.strftime('%Y-%m') for d in eixo_x_ticks], rotation=45, ha='right')

# ---- Caixa de leitura + correlações (para quem não conhece os dados) ----
texto_guia = (
    "Como ler: painel de cima mostra o quanto o MRE fala de tecnologia/regulação;\n"
    "painel de baixo mostra para que lado (China ou Ocidente) essa atenção pende.\n"
    f"Correlação Gap × Regulatório: r={r_reg:.2f} (p={p_reg:.3f})   |   "
    f"Correlação Gap × Tecnológico: r={r_tec:.2f} (p={p_tec:.3f})"
)
fig.text(0.5, -0.02, texto_guia, ha='center', fontsize=9.5, style='italic')

plt.tight_layout()
plt.show()

In [ ]:
def evolucao_bloco_termos_trimestral(lista_termos, onde='completo'):
    """Soma as menções de um grupo de termos por trimestre."""
    coluna = {'titulo': 'titulo', 'texto': 'texto', 'completo': 'texto_completo'}[onde]

    # Cria uma máscara que é verdadeira se QUALQUER um dos termos estiver presente
    padrao = '|'.join(lista_termos)
    mascara = df[coluna].str.contains(padrao, case=False, na=False, regex=True)

    # Agrupa por trimestre ('Q')
    return df[mascara].groupby(df['data_dt'].dt.to_period('Q')).size()

# Definição dos blocos de termos
blocos = {
    'TERMOS TECNOLÓGICOS': ['inteligência artificial', ' IA ', 'digital', 'infraestrutura digital', 'tecnologia'],
    'TERMOS REGULATÓRIOS': ['governança', 'padronização', 'regulação', 'cooperação', 'sul-sul'],
    'TERMOS DE PAÍSES': ['China', 'BRICS']
}

fig, ax = plt.subplots(figsize=(11, 5))

for nome_bloco, lista in blocos.items():
    serie = evolucao_bloco_termos_trimestral(lista)
    if len(serie) > 0:
        serie.plot(ax=ax, marker='o', label=nome_bloco)

ax.set_title('Evolução Trimestral por Blocos Temáticos (no texto completo)', fontsize=13)
ax.set_xlabel('Trimestre')
ax.set_ylabel('Número de notas')
ax.legend()
ax.grid(linestyle='--', alpha=0.5)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 7. ☁️ Nuvem de palavras (word cloud)

Forma visual e divertida de ver os termos mais frequentes — quanto maior a palavra, mais ela aparece.

> ⚠️ Precisa instalar a biblioteca `wordcloud` (rode a próxima célula uma vez).


In [ ]:
!pip install wordcloud -q


In [ ]:
from wordcloud import WordCloud

# Junta todos os títulos em um único texto gigante
texto_titulos = ' '.join(df['titulo'].dropna()).lower()

# Remove stopwords usando o set já definido
wc = WordCloud(
    width=1000,
    height=500,
    background_color='white',
    stopwords=STOPWORDS_PT,
    colormap='viridis',
    min_word_length=4,
    collocations=False,  # evita repetir bigramas
).generate(texto_titulos)

fig, ax = plt.subplots(figsize=(13, 6))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('Nuvem de palavras dos títulos das notas', fontsize=14)
plt.tight_layout()
plt.show()


## 8. 📏 Tamanho das notas

Quanto texto o MRE costuma escrever em cada nota? Algumas notas são bem curtas (ex: cumprimentos), outras são longas (declarações conjuntas, etc.).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histograma de quantidade de parágrafos
df['qtd_paragrafos'].plot(kind='hist', bins=20, ax=axes[0], color='cornflowerblue', edgecolor='white')
axes[0].set_title('Distribuição: parágrafos por nota')
axes[0].set_xlabel('Número de parágrafos')
axes[0].set_ylabel('Quantidade de notas')
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

# Histograma de tamanho do texto (em caracteres)
df['tamanho_texto'].plot(kind='hist', bins=20, ax=axes[1], color='salmon', edgecolor='white')
axes[1].set_title('Distribuição: tamanho do texto (caracteres)')
axes[1].set_xlabel('Caracteres')
axes[1].set_ylabel('Quantidade de notas')
axes[1].grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print("\n📊 Estatísticas:")
print(df[['qtd_paragrafos', 'tamanho_texto']].describe().round(1))


## 9. 🔗 Co-ocorrência de países

Quais países aparecem **juntos** numa mesma nota? Isso revela parcerias diplomáticas e contextos compartilhados (ex: declarações conjuntas).


In [ ]:
import numpy as np

paises_analise = ['China', 'Estados Unidos', 'Europa', 'Latina', 'Ásia', 'Sul', 'BRICS', 'União Europeia', 'CELAC',
                  'Digital', 'Tecnologia', 'Inteligência Artificial', ' IA ', 'Governança']

# Para cada nota, marca quais países são mencionados
matriz_presenca = pd.DataFrame({
    pais: df['texto_completo'].str.contains(pais, case=False, na=False, regex=False).astype(int)
    for pais in paises_analise
})

# Matriz de co-ocorrência (multiplicação matricial: notas onde A E B aparecem)
coocorrencia = matriz_presenca.T.dot(matriz_presenca)

# Zerar a diagonal (não interessa um país com ele mesmo)
np.fill_diagonal(coocorrencia.values, 0)

# Heatmap simples com matplotlib
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(coocorrencia, cmap='YlOrRd', aspect='auto')

ax.set_xticks(range(len(paises_analise)))
ax.set_yticks(range(len(paises_analise)))
ax.set_xticklabels(paises_analise, rotation=45, ha='right')
ax.set_yticklabels(paises_analise)
ax.set_title('Co-ocorrência: quantas notas mencionam dois países juntos', fontsize=13)

# Anotar os valores nas células
for i in range(len(paises_analise)):
    for j in range(len(paises_analise)):
        valor = coocorrencia.iloc[i, j]
        if valor > 0:
            ax.text(j, i, int(valor), ha='center', va='center',
                    color='black' if valor < coocorrencia.values.max()/2 else 'white',
                    fontsize=9)

plt.colorbar(im, ax=ax, label='Nº de notas em comum')
plt.tight_layout()
plt.show()


---

## 10. 🤖 Tecnologia/IA: qual enquadramento (frame) predomina nas notas do MRE?

Já vimos a evolução temporal de blocos temáticos. Agora vamos olhar só para as notas
que falam de **tecnologia, digital ou inteligência artificial** e perguntar: dentro
delas, o vocabulário usado está mais próximo de qual enquadramento?

1. **Regulação & Direitos** — regulação, compliance, direitos fundamentais, transparência, accountability, risco, auditoria, fiscalização, proteção de dados, LGPD, AI Act, GDPR
2. **Desenvolvimento & Inclusão** — desenvolvimento, capacitação, inclusão digital, cooperação Sul-Sul, acesso, infraestrutura, inovação, indústria nacional, competitividade, autonomia tecnológica
3. **Soberania & Segurança** — soberania digital, segurança cibernética, independência tecnológica, dependência, vulnerabilidade, controle, segurança nacional, dado estratégico
4. **Investimento & Negócios** — investimento, parceria estratégica, negócios, comércio, retorno econômico, PIB, emprego, indústria 4.0, produtividade, cadeia de valor

Cada enquadramento tem um dicionário de termos (os termos abaixo são um ponto de
partida — ajuste a lista livremente). Contamos quantas vezes cada termo aparece no
**texto completo** das notas de tecnologia e comparamos os quatro blocos.


In [ ]:
import json
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 0. Carrega e prepara os dados do zero -----------------------------------
# Célula autossuficiente: não depende de nenhuma célula anterior ter rodado.
with open('nota_mre.json', encoding='utf-8') as f:
    _raw = json.load(f)
df_base = pd.DataFrame.from_dict(_raw['_default'], orient='index')
df_base['texto'] = df_base['paragrafo'].apply(lambda lista: ' '.join(lista) if isinstance(lista, list) else str(lista))
df_base['texto_completo'] = (df_base['titulo'].fillna('') + ' ' + df_base['texto'].fillna('')).str.lower()

# 1. Filtrar notas relacionadas a tecnologia / digital / IA -----------------
termos_filtro_tech = [
    r'intelig[êe]ncia\s+artificial', r'\bIA\b', r'\bAI\b',
    r'\bdigital\b', r'tecnolog', r'cyberseguran', r'ciberseguran',
]
padrao_tech = '|'.join(termos_filtro_tech)
mask_tech = df_base['texto_completo'].str.contains(padrao_tech, case=False, na=False, regex=True)
df_tech = df_base[mask_tech].copy()
print(f"Notas relacionadas a tecnologia/digital/IA: {len(df_tech)} de {len(df_base)} ({len(df_tech)/len(df_base)*100:.1f}%)")

# 2. Os quatro enquadramentos (frames) e seus termos -------------------------
# Cores seguem uma paleta categórica fixa (mesma ordem sempre = mesma cor por bloco)
categorias = {
    'Regulação & Direitos': {
        'cor': '#2a78d6',
        'termos': [
            'regulaç', 'compliance', 'direitos fundamentais', 'direitos humanos',
            'transparência', 'accountability', 'responsabilização', 'risco',
            'auditoria', 'fiscalização', 'proteção de dados', 'LGPD', 'AI Act',
            'GDPR', 'privacidade', 'ética',
        ],
    },
    'Desenvolvimento & Inclusão': {
        'cor': '#1baf7a',
        'termos': [
            # 'desenvolvimento', 
            'capacitação', 'inclusão digital', 'cooperação sul-sul',
            'sul-sul', 
            # 'acesso', 
            'infraestrutura', 
            # 'inovação', 
            'indústria nacional',
            'competitividade', 'autonomia tecnológica', 'transferência de tecnologia',
        ],
    },
    'Soberania & Segurança': {
        'cor': '#eda100',
        'termos': [
            'soberania digital', 'soberania', 'segurança cibernética', 'cibersegurança',
            'independência tecnológica', 'dependência', 'vulnerabilidade', 'controle',
            'segurança nacional', 'dado estratégico', 'dados estratégicos',
        ],
    },
    # 'Investimento & Negócios': {
    #     'cor': '#008300',
    #     'termos': [
    #         'investimento', 'parceria estratégica', 'negócios', 'comércio',
    #         'retorno econômico', 'PIB', 'emprego', 'indústria 4.0', 'produtividade',
    #         'cadeia de valor', 'exportação', 'mercado',
    #     ],
    # },
}

# 3. Contar ocorrências de cada termo dentro das notas de tecnologia --------
linhas = []
for nome_cat, info in categorias.items():
    for termo in info['termos']:
        padrao = rf"(?i)\b{re.escape(termo.strip())}\b"
        contagem = df_tech['texto_completo'].str.count(padrao).sum()
        if contagem > 0:
            linhas.append({'categoria': nome_cat, 'termo': termo, 'contagem': int(contagem), 'cor': info['cor']})

termos_df = pd.DataFrame(linhas).sort_values('contagem', ascending=False)
resumo_cat = termos_df.groupby('categoria')['contagem'].sum()

# 4. Gráfico: termos mais frequentes (esq.) + total por enquadramento (dir.) -
top_termos = termos_df.head(20).sort_values('contagem')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8), gridspec_kw={'width_ratios': [1.3, 1]})

# Painel 1 — top termos, coloridos pelo enquadramento a que pertencem
ax1.barh(top_termos['termo'], top_termos['contagem'], color=top_termos['cor'])
ax1.set_title(f'Principais termos em notas de tecnologia/digital/IA\n({len(df_tech)} notas)', fontsize=12)
ax1.set_xlabel('Ocorrências no texto completo')
ax1.grid(axis='x', linestyle='--', alpha=0.4)
for i, valor in enumerate(top_termos['contagem']):
    ax1.text(valor + top_termos['contagem'].max() * 0.01, i, str(valor), va='center', fontsize=8, color='#52514e')

handles = [mpatches.Patch(color=info['cor'], label=nome) for nome, info in categorias.items()]
ax1.legend(handles=handles, loc='lower right', fontsize=8, framealpha=0.9)

# Painel 2 — total de ocorrências por enquadramento
ordem_cat = list(categorias.keys())
valores_cat = resumo_cat.reindex(ordem_cat).fillna(0)
cores_cat = [categorias[c]['cor'] for c in ordem_cat]
pct_cat = valores_cat / valores_cat.sum() * 100

barras = ax2.bar(range(len(ordem_cat)), valores_cat.values, color=cores_cat)
ax2.set_xticks(range(len(ordem_cat)))
ax2.set_xticklabels([c.replace(' & ', '\n& ') for c in ordem_cat], fontsize=9)
ax2.set_title('A qual enquadramento as notas de\ntecnologia estão mais associadas?', fontsize=12)
ax2.set_ylabel('Total de ocorrências dos termos')
ax2.grid(axis='y', linestyle='--', alpha=0.4)

for barra, valor, pct in zip(barras, valores_cat.values, pct_cat.values):
    ax2.text(barra.get_x() + barra.get_width() / 2, valor + valores_cat.max() * 0.01,
              f'{int(valor)}\n({pct:.0f}%)', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

# 5. Resumo em texto ----------------------------------------------------------
print("\nResumo por enquadramento (ocorrências no texto completo das notas de tecnologia):")
for cat in ordem_cat:
    print(f"  {cat:30s} {int(valores_cat[cat]):5d}  ({pct_cat[cat]:.1f}%)")

print(
    "\nAtenção: 'desenvolvimento' e 'comércio' são palavras muito genéricas no "
    "vocabulário diplomático (aparecem em contextos que não são só tecnológicos, "
    "como 'desenvolvimento sustentável' ou 'comércio bilateral'). Isso infla o bloco "
    "Desenvolvimento & Inclusão / Investimento & Negócios. Para uma leitura mais "
    "conservadora, veja `termos_df` e desconsidere os termos mais genéricos, ou "
    "compare pelo percentual de NOTAS (não de ocorrências) que citam cada bloco:"
)
for nome_cat, info in categorias.items():
    padrao = '|'.join(rf"\b{re.escape(t.strip())}\b" for t in info['termos'])
    n_notas = df_tech['texto_completo'].str.contains(padrao, case=False, na=False, regex=True).sum()
    print(f"  {nome_cat:30s} {n_notas:4d} notas ({n_notas/len(df_tech)*100:.1f}% das notas de tecnologia)")


---

## 🎯 O que mais você pode fazer?

Algumas ideias para continuar explorando:

- **Extração de entidades nomeadas** (NER) com `spaCy` para encontrar países, pessoas e organizações automaticamente, sem ter que listar tudo na mão
- **Análise de sentimento** dos textos (uma nota é mais "neutra", "preocupada", "celebrativa"?)
- **Tópicos latentes** com LDA ou BERTopic para descobrir agrupamentos temáticos
- **Filtro interativo** com `ipywidgets` para o usuário digitar um termo e ver o resultado em tempo real
- **Salvar os gráficos** como PNG/PDF com `plt.savefig('nome.png', dpi=200, bbox_inches='tight')`
- **Exportar buscas** para Excel: `buscar_termo('Israel').to_excel('resultado.xlsx', index=False)`
